# Sequence Element Classification

* We will study how a *transformer* architecture can be used for sequence element classification.

* A sequence element classification dataset is a sequence of examples. Each example is a pair of sequences of vectors. Both sequences in an example have the same number of elements.

* For instance, if the first element of an example is a sequence of vectors $\mathbf{x}_{1}, \ldots, \mathbf{x}_{T}$ with $T$ elements, where $\mathbf{x}_t \in \mathbb{R}^d$, then the second element of the same example may be a sequence of (one-hot encoded) vectors $\mathbf{y}_1, \ldots, \mathbf{y}_{T}$, where $\mathbf{y}_t \in \mathbb{R}^q$.

* If the example above is part of the training dataset, a sequence element classification model would attempt to predict the target vector $\mathbf{y}_t \in \mathbb{R}^q$ given the sequence of input vectors $\mathbf{x}_1, \ldots, \mathbf{x}_t$, for every $t$. In this context, $t$ is called the number of *time steps*.

* As always, the objective is to find a a model that generalizes well (makes good predictions for unseen input sequences).

* Current large language models are based on transformer architectures trained for sequence element classification.


# Self-Attention: Overview

* A masked scaled dot-product self-attention layer is the most distinctive type of layer in a typical transformer architecture trained for sequence element classification.

* **We will use the term self-attention to refer to masked scaled dot-product self-attention.**

* A self-attention layer receives an input sequence of vectors $\mathbf{x}_{1}, \ldots, \mathbf{x}_{T}$ with $T$ elements and produces an output sequence of vectors $\mathbf{h}_{1}, \ldots, \mathbf{h}_{T}$ with $T$ elements.

* For every $t$, the self-attention layer uses the input $\mathbf{x}_{t}$ to produce a query $\mathbf{q}_t$, a key $\mathbf{k}_t$, and a value $\mathbf{v}_t$.

* For every $t$, the output $\mathbf{h}_{t}$ is an *attention*-weighted sum of the values $\mathbf{v}_{1}, \ldots, \mathbf{v}_{t}$. These *attention weights* are derived from the *affinity* between the query $\mathbf{q}_{t}$ and the keys $\mathbf{k}_1, \ldots, \mathbf{k}_{t}$.

* The parameters of the self-attention layer are adapted so that the output $\mathbf{h}_{t}$ summarizes the sequence $\mathbf{x}_{1}, \ldots, \mathbf{x}_{t}$ for the purposes of the sequence element classification task, for every $t$.

* The term *self*-attention comes from the fact that queries, keys, and values are derived from the same input sequence.

* A *cross-attention* layer receives two input sequences. The queries are derived from the first input sequence while the keys and values are derived from the second input sequence.


# Self-Attention: Forward Pass

* Consider a self-attention layer that receives a sequence of observations $\mathbf{x}_{1}, \ldots, \mathbf{x}_{T}$ with $T$ elements, where $\mathbf{x}_t \in \mathbb{R}^d$.

* Suppose that this input sequence is organized into a matrix  $\mathbf{X} \in \mathbb{R}^{T \times d}$, so that the $t$-th row of $\mathbf{X}$ corresponds to the (transposed) observation $\mathbf{x}_{t}$, for every $t$.

* **This notation is fundamentally incompatible with the notation that we employed for recurrent neural networks.**

* A self-attention layer has two hyperparameters: the query dimension $a$ and the output dimension $h$.

* The query matrix $\mathbf{Q} \in \mathbb{R}^{T \times a}$ is given by

$$ \mathbf{Q} = \mathbf{X} \mathbf{W}^{(Q)} + \mathbf{B}^{(Q)},$$

where $\mathbf{W}^{(Q)} \in \mathbb{R}^{d \times a}$ and $\mathbf{B}^{(Q)} \in \mathbb{R}^{T \times a}$ are parameter matrices, and the parameter matrix $\mathbf{B}^{(Q)}$ is obtained by transposing and replicating the same parameter vector $\mathbf{b}^{(Q)} \in \mathbb{R}^{a}$ across $T$ rows.

* The query $\mathbf{q}_t \in \mathbb{R}^{a}$ for observation $\mathbf{x}_t$ is obtained by transposing the $t$-th row of $\mathbf{Q}$, for every $t$.

* The key matrix $\mathbf{K} \in \mathbb{R}^{T \times a}$ is given by

$$ \mathbf{K} = \mathbf{X} \mathbf{W}^{(K)} + \mathbf{B}^{(K)},$$

where $\mathbf{W}^{(K)} \in \mathbb{R}^{d \times a} $ and $\mathbf{B}^{(K)} \in \mathbb{R}^{T \times a}$ are parameter matrices, and the parameter matrix $\mathbf{B}^{(K)}$ is obtained by transposing and replicating the same parameter vector $\mathbf{b}^{(K)} \in \mathbb{R}^{a}$ across $T$ rows.

* The key $\mathbf{k}_t \in \mathbb{R}^{a}$ for observation $\mathbf{x}_t$ is obtained by transposing the $t$-th row of $\mathbf{K}$, for every $t$.

* The value matrix $\mathbf{V} \in \mathbb{R}^{T \times h}$ is given by

$$ \mathbf{V} = \mathbf{X} \mathbf{W}^{(V)} + \mathbf{B}^{(V)}, $$
where $\mathbf{W}^{(V)} \in \mathbb{R}^{d \times h} $ and $\mathbf{B}^{(V)} \in \mathbb{R}^{T \times h}$ are parameter matrices, and the parameter matrix $\mathbf{B}^{(V)}$ is obtained by transposing and replicating the same parameter vector $\mathbf{b}^{(V)} \in \mathbb{R}^{h}$ across $T$ rows.

* The value $\mathbf{v}_t \in \mathbb{R}^{h}$ for observation $\mathbf{x}_t$ is obtained by transposing the $t$-th row of $\mathbf{V}$, for every $t$.

* The score matrix $\mathbf{S} \in \mathbb{R}^{T \times T}$ is given by

$$ \mathbf{S} = \frac{\mathbf{Q} \mathbf{K}^T}{\sqrt{a}}.$$

* If $S_{i,j}$ denotes the $(i,j)$-th element of the score matrix $\mathbf{S}$, then $ S_{i, j} = \mathbf{q}_{i}^{T} \mathbf{k}_{j} / \sqrt{a}$.

* Intuitively, $S_{i,j}$ is the *affinity* (alignment) between the query $\mathbf{q}_{i}$ and the key $\mathbf{k}_{j}$.

* In other words, the $i$-th row of the score matrix $\mathbf{S}$ contains the *affinity* between the query $\mathbf{q}_{i}$ and the keys $\mathbf{k}_{1}, \ldots, \mathbf{k}_{T}$.

* The masked score matrix $\mathbf{S}' \in \mathbb{R}^{T \times T}$ is given by
$$ S_{i, j}' =
    \begin{cases}
        S_{i, j}, &\text{ if } j \leq i, \\
        - \infty, &\text{ if } j > i.
    \end{cases}
$$

* Intuitively, masking destroys information about the affinity between a query $\mathbf{q}_i$ and every future key $\mathbf{k}_{i + 1}, \ldots, \mathbf{k}_{T}$, for every $i$. As will become clear, this will prevent the output for a given time step from depending on information that will only be available at a later time step.

* The attention matrix $\mathbf{A} \in \mathbb{R}^{T \times T}$ is given by

$$
\mathbf{A} = \text{softmax}(\mathbf{S}'),
$$
where the $\text{softmax}$ function is applied individually to each row of the masked score matrix $\mathbf{S}'$.

* The attention $\mathbf{a}_{t} \in \mathbb{R}^T$ for observation $\mathbf{x}_t$ is obtained by transposing the $t$-th row of $\mathbf{A}$, for every $t$.

* Note that $A_{i, j} = a_{i, j} = 0$ whenever $j > i$.

* The output matrix $\mathbf{H} \in \mathbb{R}^{T \times h}$ is given by

$$ \mathbf{H} =  \mathbf{A} \mathbf{V}. $$

* The output $\mathbf{h}_t \in \mathbb{R}^{h}$ for observation $\mathbf{x}_t$ is obtained by transposing the $t$-th row of $\mathbf{H}$, for every $t$. Therefore,

$$ \mathbf{h}_t = \sum_{k = 1}^{t} a_{t, k} \mathbf{v}_k. $$

* Intuitively, the output $\mathbf{h}_{t}$ for observation $\mathbf{x}_{t}$ is an attention-weighted sum of the values $\mathbf{v}_{1}, \ldots, \mathbf{v}_{t}$. These attention weights are derived from the affinity between the query $\mathbf{q}_{t}$ and the keys $\mathbf{k}_1, \ldots, \mathbf{k}_{t}$.


# Self-Attention: Training

* Consider an example composed of a sequence of observations $\mathbf{x}_{1}, \ldots, \mathbf{x}_{T}$ with $T$ elements, where $\mathbf{x}_t \in \mathbb{R}^d$, and a sequence of (one-hot encoded) targets $\mathbf{y}_1, \ldots, \mathbf{y}_{T}$, where $\mathbf{y}_t \in \mathbb{R}^q$.

* Suppose that the sequence of observations is organized into a matrix $\mathbf{X} \in \mathbb{R}^{T \times d}$, so that the $t$-th row of $\mathbf{X}$ corresponds to the (transposed) observation $\mathbf{x}_{t}$, for every $t$.

* Suppose that the sequence of targets is organized into a matrix  $\mathbf{Y} \in \mathbb{R}^{T \times q}$, so that the $t$-th row of $\mathbf{Y}$ corresponds to the (transposed) target $\mathbf{y}_{t}$.

* Let $\mathbf{H} \in \mathbb{R}^{T \times q}$ denote the output matrix of a self-attention layer with output dimension $h = q$ upon receiving the matrix $\mathbf{X}$.

* A prediction matrix $\hat{\mathbf{Y}} \in \mathbb{R}^{T \times q}$ can be computed by

$$ \hat{\mathbf{Y}} = \text{softmax}(\mathbf{H}), $$

where the softmax function is applied individually to each row of the (logits) matrix $\mathbf{H}$.

* Let $l(\hat{\mathbf{Y}}, \mathbf{Y})$ denote the cross-entropy loss between a prediction matrix $\hat{\mathbf{Y}} \in \mathbb{R}^{T \times q}$ and a target matrix $\mathbf{Y} \in \mathbb{R}^{T \times q}$.

* If a sequence element classification dataset composed of $n$ examples is organized into a sequence of pairs of matrices $(\mathbf{X}^{(1)}, \mathbf{Y}^{(1)}), \ldots, (\mathbf{X}^{(n)}, \mathbf{Y}^{(n)})$, a self-attention layer could be trained by minimizing the average loss across examples and time steps given by

$$ \frac{1}{nT} \sum_{i = 1}^{n} l(\mathbf{\hat{Y}}^{(i)}, \mathbf{Y}^{(i)}), $$

where each prediction matrix $\mathbf{\hat{Y}}^{(i)}$ depends on the parameters of the self-attention layer and the matrix $\mathbf{X}^{(i)}$.

* Note that the input matrices could be further organized into a single $n \times T \times d$ tensor $[\mathbf{X}^{(1)}, \ldots, \mathbf{X}^{(n)}]$. Similarly, the target matrices could be organized into a single $n \times T \times q$ tensor $[\mathbf{Y}^{(1)}, \ldots, \mathbf{Y}^{(n)}]$.

* However, self-attention layers are rarely employed on their own. Instead, they are typically part of a so-called *transformer* layer.

* Before presenting a typical transformer layer, we will introduce positional encoding, multi-headed self-attention, and layer normalization.

# Positional Encoding

* Let $\mathbf{h}_1, \ldots, \mathbf{h}_{T}$, where $\mathbf{h}_{t} \in \mathbb{R}^{h}$, denote the output of a self-attention layer with output dimension $h$ upon receiving a sequence of observations $\mathbf{x}_{1}, \ldots, \mathbf{x}_{T}$, where $\mathbf{x}_t \in \mathbb{R}^d$.

* Recall that the output $\mathbf{h}_{T}$ for observation $\mathbf{x}_{T}$ is an attention-weighted sum of the values $\mathbf{v}_{1}, \ldots, \mathbf{v}_{T}$. These attention weights are derived from the affinity between the query $\mathbf{q}_{T}$ and the keys $\mathbf{k}_1, \ldots, \mathbf{k}_{T}$.

* If the elements of the sequence $\mathbf{x}_{1}, \ldots, \mathbf{x}_{T-1}$ are permuted, then the output $\mathbf{h}_{T}$ remains the same.

* This is generally undesirable in sequence element classification and commonly addressed by employing a positional encoding layer.

* A positional encoding layer has a single hyperparameter: the maximum sequence length $M$.

* Suppose that the sequence of observations $\mathbf{x}_{1}, \ldots, \mathbf{x}_{T}$ is organized into a matrix  $\mathbf{X} \in \mathbb{R}^{T \times d}$, so that the $t$-th row of $\mathbf{X}$ corresponds to the (transposed) observation $\mathbf{x}_{t}$, for every $t$.

* The output matrix $\mathbf{X}' \in \mathbb{R}^{T \times d}$ of a positional encoding layer is given by

$$ \mathbf{X}' = \mathbf{X} + \mathbf{P}_{1:T}, $$

where $\mathbf{P}_{1:T} \in \mathbb{R}^{T \times d}$ denotes a matrix composed of the first $T$ rows of a parameter matrix $\mathbf{P} \in \mathbb{R}^{M \times d}$.

* The output matrix $\mathbf{X}'$ of a positional encoding layer is processed by the next layer.

* Intuitively, each row of the parameter matrix $\mathbf{P}$ can provide a code to represent a *time step* if such information is helpful for the sequence element classification task. Naturally, appropriate codes are learned by gradient descent.

* Sequences with more than $M$ elements have to be truncated (by discarding their earliest elements) in order to apply positional encoding.

* In principle, alternative positional encoding implementations are able to deal with arbitrary sequence lengths.

* In practice, what limits the maximum sequence length is the computational cost of backpropagation for self-attention layers that multiply a $T \times T$ attention matrix by a value matrix.


# Multi-Headed Self-Attention

* In a multi-headed self-attention layer, a single input sequence is processed by multiple independent self-attention layers whose output sequences are combined to produce a single output sequence. Each of these self-attention layers is called an *attention head*.

* Intuitively, the parameters of an attention head are adapted so that its output for every given time step summarizes a different aspect of the input sequence up to that time step.

* Consider a multi-headed self-attention layer that receives a sequence of observations $\mathbf{x}_{1}, \ldots, \mathbf{x}_{T}$ with $T$ elements, where $\mathbf{x}_t \in \mathbb{R}^d$.

* Suppose that the sequence of observations is organized into a matrix $\mathbf{X} \in \mathbb{R}^{T \times d}$, so that the $t$-th row of $\mathbf{X}$ corresponds to the (transposed) observation $\mathbf{x}_{t}$, for every $t$.

* A multi-headed self-attention layer has four hyperparameters: the number of heads $c$, the output dimension $o$, the query dimension $a$, and the head output dimension $h$.

* A total of $c$ self-attention layers with query dimension $a$ and output dimension $h$ receive the same input matrix $\mathbf{X}$ and independently produce the output matrices $\mathbf{H}_{1}, \ldots, \mathbf{H}_{c}$, where $\mathbf{H}_{i} \in \mathbb{R}^{T \times h}$.

* The output matrix $\mathbf{H} \in \mathbb{R}^{T \times o}$ of the multi-headed self attention layer is obtained by concatenating the output matrices of the $c$ individual self-attention layers across rows and performing a simple transformation:
$$\mathbf{H} = [ \mathbf{H}_1, \ldots, \mathbf{H}_{c} ] \mathbf{W} + \mathbf{B},$$
where $\mathbf{W} \in \mathbb{R}^{ch \times o}$ and $\mathbf{B} \in \mathbb{R}^{T \times o }$ are parameter matrices of the multi-headed self attention layer, and the matrix $\mathbf{B}$ is obtained by transposing and replicating the same parameter vector $\mathbf{b} \in \mathbb{R}^{o}$ across $T$ rows.

* It is common to choose a number of heads $c$ that divides the input dimension $d$ and let the remaining hyperparameters be given by $o = d$ and $a = h = d/c$. As a consequence, the input matrix $\mathbf{X}$ and the output matrix $\mathbf{H}$ will have the same shape.



# Layer Normalization


* A layer normalization receives a sequence of vectors $\mathbf{x}_{1}, \ldots, \mathbf{x}_{T}$ with $T$ elements, where $\mathbf{x}_{t} \in \mathbb{R}^d$, and produces a sequence of vectors $\mathbf{x}'_{1}, \ldots, \mathbf{x}'_{T}$, where $\mathbf{x}_{t}' \in \mathbb{R}^d$.

* For every $t$, the mean $\mu_{t}$ and variance $\sigma^{2}_t$ are given by

$$ \mu_{t} = \frac{1}{d}\sum_{i = 1}^d x_{t, i},$$

and

$$ \sigma_{t}^2 = \frac{1}{d} \sum_{i =1}^d (x_{t, i} - \mu_{t})^2. $$

* For every $t$, the output vector $\mathbf{x}_{t}'$ is given by

$$ \mathbf{x}_{t}' = \frac{\mathbf{x}_{t} - \mu_{t}}{\sigma_t + \epsilon} \odot \boldsymbol{\gamma} + \boldsymbol{\beta}, $$

where $\boldsymbol{\gamma} \in \mathbb{R}^{d}$ and $\boldsymbol{\beta} \in \mathbb{R}^d$ are parameter vectors and $\epsilon > 0$ is a small constant that prevents division by zero.

* If the sequence of inputs is organized into a matrix $\mathbf{X} \in \mathbb{R}^{T \times d}$ where the $t$-th row of $\mathbf{X}$ corresponds to the (transposed) vector $\mathbf{x}_{t}$, layer normalization produces a matrix $\mathbf{X}' \in \mathbb{R}^{T \times d}$ where the $t$-th row of $\mathbf{X}'$ corresponds to the (transposed) vector $\mathbf{x}_{t}'$, for every $t$.


# Generative Pre-trained Transformer (GPT): Overview

* The Generative Pre-trained Transformer (GPT) architecture is a *decoder* transformer architecture, which is appropriate for sequence element classification. There are also *encoder* transformer architectures and *encoder-decoder* transformer architectures.

* We will present a (slightly adapted) version of the GPT architecture as an illustration of a typical transformer architecture.

* The GPT architecture employs a sequence of *transformer layers*.

* A typical transformer layer passes an input sequence through a multi-headed self-attention layer, a first layer normalization, two fully-connected layers, and a second layer normalization. It also employs residual connections, as illustrated below.

![Transformer layer (adapted from "Understanding Deep Learning")](https://drive.google.com/uc?export=view&id=10cUu8h_NjNGoFgaUF9e2I3Ecn7qZ99_R)

* **The exact composition of a transformer layer varies across implementations.** In what follows, the order of the layers within a transformer layer is slightly different than illustrated above.

# Generative Pre-trained Transformer (GPT): Forward Pass

* **Hyperparameters:**
    * The input dimension $d$.

    * The number of classes $q$.

    * The maximum sequence length $M$.

    * The number of transformer layers $L$.

    * The number of heads $c$, which should divide the input dimension $d$.
        * The number of heads $c$, the output dimension $o = d$, the query dimension $a = d/c$, and the head output dimension $h = d/c$ are fixed for every multi-headed self-attention layer in the architecture.

    * The number of units $u$ in the first fully-connected layer witin each transformer layer. Typically, $u = 4d$.

* **Input:** an input matrix  $\mathbf{X} \in \mathbb{R}^{T \times d}$, where $T \leq M$.

* **Output:** a prediction matrix $\hat{\mathbf{Y}} \in \mathbb{R}^{T \times q}$.

* **Algorithm:**

    1. $ \mathbf{X} \gets $ the output of positional encoding applied to $\mathbf{X}$ with a parameter matrix $\mathbf{P} \in \mathbb{R}^{M \times d}$.

    1. For each transformer layer $l \in 1, \ldots, L$:

        1. $\mathbf{X}' \gets$ the output of layer normalization applied to $\mathbf{X}$ with parameters vectors $\boldsymbol{\gamma}^{(l, 1)} \in \mathbb{R}^{d}$ and $\boldsymbol{\beta}^{(l, 1)} \in \mathbb{R}^d$.
        1. $\mathbf{X}' \gets$ the output of multi-headed self-attention layer $l$ applied to $\mathbf{X}'$.
        1. $ \mathbf{X} \gets  \mathbf{X} + \mathbf{X}'$, a residual connection.
        1. $\mathbf{X}' \gets$ the output of layer normalization applied to $\mathbf{X}$ with parameters vectors $\boldsymbol{\gamma}^{(l, 2)} \in \mathbb{R}^{d}$ and $\boldsymbol{\beta}^{(l, 2)}\in \mathbb{R}^d$.
        1. $\mathbf{X} \gets \mathbf{X} + \text{ReLU}( \mathbf{X}'\mathbf{W}^{(l,1)}  + \mathbf{B}^{(l,1)})\mathbf{W}^{(l,2)} + \mathbf{B}^{(l,2)}$, where $\mathbf{W}^{(l,1)} \in \mathbb{R}^{d \times u}$ and $\mathbf{W}^{(l,2)} \in \mathbb{R}^{u \times d}$ are weight matrices and $\mathbf{B}^{(l,1)} \in \mathbb{R}^{T \times u}$ and $\mathbf{B}^{(l,2)} \in \mathbb{R}^{T \times d}$ are bias matrices (with replicated rows).

    1. $\mathbf{X} \gets $ the output of layer normalization applied to $\mathbf{X}$ with parameters vectors $\boldsymbol{\gamma} \in \mathbb{R}^{d}$ and $\boldsymbol{\beta}\in \mathbb{R}^d$.

    1. $\mathbf{H} \gets \mathbf{X} \mathbf{W} $, where $\mathbf{W} \in \mathbb{R}^{d \times q}$ is a weight matrix.

    1. $\hat{\mathbf{Y}} \gets \text{softmax}(\mathbf{H})$, where the softmax function is applied individually to each row of the logits matrix $\mathbf{H}$.

# Generative Pre-Trained Transformer (GPT): Training

* Let $l(\hat{\mathbf{Y}}, \mathbf{Y})$ denote the cross-entropy loss between a prediction matrix $\hat{\mathbf{Y}} \in \mathbb{R}^{T \times q}$ and a target matrix $\mathbf{Y} \in \mathbb{R}^{T \times q}$.

* Suppose that a sequence element classification dataset composed of $n$ examples is organized into a sequence of pairs of matrices $(\mathbf{X}^{(1)}, \mathbf{Y}^{(1)}), \ldots, (\mathbf{X}^{(n)}, \mathbf{Y}^{(n)})$.

* For simplicity, suppose that each example in the dataset is composed of two sequences of length $T$.

* A transformer can be trained by minimizing the average loss across examples and time steps given by

$$ \frac{1}{nT} \sum_{i = 1}^{n} l(\mathbf{\hat{Y}}^{(i)}, \mathbf{Y}^{(i)}), $$

where each prediction matrix $\mathbf{\hat{Y}}^{(i)} \in \mathbb{R}^{T \times q}$ depends on the parameters of the model and the input matrix $\mathbf{X}^{(i)} \in \mathbb{R}^{T \times d}$.

* Note that the input matrices could be further organized into a single $n \times T \times d$ tensor $[\mathbf{X}^{(1)}, \ldots, \mathbf{X}^{(n)}]$. Similarly, the target matrices could be organized into a single $n \times T \times q$ tensor $[\mathbf{Y}^{(1)}, \ldots, \mathbf{Y}^{(n)}]$.


# GPT-Based Language Models


* Recall that a language model assigns a probability to each possible sequence of tokens (text).

* By the chain rule of probability, for every sequence of tokens $x_1, \ldots, x_T$, a language model only needs to assign a probability $\mathbb{P}(X_1 = x_1)$ to $x_1$ being the first token and, for every $t > 1$, a probability $\mathbb{P}(X_{t} = x_{t} \mid X_1 = x_1, \ldots, X_{t-1} = x_{t-1})$ to $x_t$ being the $t$-th token if the previous tokens are $x_{1}, \ldots, x_{t-1}$.

* An autoregressive language model can be used to generate text by sampling the first token $x_1$ from the distribution for $X_1$ and, for every $t > 1$, sampling the token $x_t$ from the distribution for $X_t$ given $X_1 = x_1, \ldots, X_{t-1} = x_{t-1}$.

* A Generative Pre-trained Transformer trained to predict the next (one-hot encoded) token $\mathbf{y}_{t} = \mathbf{x}_{t+1}$ given a sequence of (one-hot encoded) tokens $\mathbf{x}_1, \ldots, \mathbf{x}_{t}$ can be used as an autoregressive language model, since its prediction vector $\hat{\mathbf{y}}_{t}$ can be interpreted as a probability distribution over the next token given the previous tokens.


# Large Language Models


* [GPT-3](https://arxiv.org/abs/2005.14165) was the first large language model to receive widespread attention, although [GPT-2](https://openai.com/index/better-language-models/) already exhibited remarkable *in-context learning* capabilities, which surprised most machine learning researchers.

* GPT-3 employs a large GPT-like architecture:
    * The input dimension is $d = 12288$.

    * The vocabulary has $q = 50257$ tokens.

    * An additional parameter matrix $\mathbf{E} \in \mathbb{R}^{q \times d}$ called *embedding matrix* provides a $d$-dimensional code vector for each of the $q$ tokens in the vocabulary.

    * The maximum sequence length is $M = 2048$.

    * The number of transformer layers is $L = 96$.

    * The number of heads is $c = 96$, which divides the input dimension $d = 12288$.
        
        * The number of heads $c = 96$, the output dimension $o = d = 12888$, the query dimension $a = d/c = 128$, and the head output dimension $h = d/c = 128$ are fixed for every multi-headed self-attention layer in the architecture.
    
    * The number of units $u$ in the first fully-connected layer witin each transformer layer is $u = 4d = 49152$.

* The GPT-3 architecture has approximately $175$ billion parameters.

* Using these hyperparameters, the GPT-like architecture described earlier would have $175221817344 \approx 175$ billion parameters.

* OpenAI trained GPT-3 using approximately $300$ billion tokens.

* It is possible to induce a large language model to perform a desired task by providing an appopriate *prompt* (sequence of tokens).

* For example, in order to query a large language model about the capital of Portugal, the prompt could provide examples of questions and answers in a desired format:

> Q. What is the capital of France?
>
> A. Paris.
>
> Q. What is the capital of Spain?
>
> A. Madrid.
>
> Q. What is the capital of Portugal?
>
> A.

* Currently, developing large language models is often divided into three stages:
    1. Pre-training: training a large transformer-based autoregressive language model on a large training dataset.
    1. Supervised fine-tuning: retraining  the resulting transformer-based autoregressive language model on a carefully crafted dataset composed of prompts and responses (which may include detailed reasoning steps).
    1. Reinforcement learning: retraining the resulting transformer-based autoregressive language model to increase the probability of desirable output sequences given a mechanism to parse and score output sequences (or human feedback).



# Recommended Reading

* [Formal Algorithms for Transformers](https://arxiv.org/abs/2207.09238).

* [Understanding Deep Learning](https://udlbook.github.io/udlbook/): Chapter 12.

* [Training and fine-tuning large language models](https://rbcborealis.com/research-blogs/training-and-fine-tuning-large-language-models).

* [Dive into Deep Learning](https://d2l.ai): Chapter 11.

* [Neural Networks](https://www.youtube.com/playlist?list=PLZHQObOWTQDNU6R1_67000Dx_ZCJB-3pi): Videos DL5-DL7.

* [Neural Networks: Zero to Hero](https://karpathy.ai/zero-to-hero.html): "Let's build GPT: from scratch, in code, spelled out".

* [DeepSeek-R1](https://arxiv.org/abs/2501.12948).
